<a href="https://colab.research.google.com/github/pariupadhyay15/code-reviewer/blob/main/code_reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/code-reviewer-llm'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
print('Project dir ready:', PROJECT_DIR)

Mounted at /content/drive
Project dir ready: /content/drive/MyDrive/code-reviewer-llm


In [4]:
from datasets import load_dataset

raw = load_dataset("ronantakizawa/github-codereview")
print(raw)
print()
print("Columns:", raw['train'].column_names)
print()
for i in [0, 1, 2]:
    ex = raw['train'][i]
    print(f"--- Example {i} ---")
    for k, v in ex.items():
        preview = str(v)[:200]
        print(f"{k}: {preview}")
    print()

README.md:   0%|          | 0.00/5.42k [00:00<?, ?B/s]

data/train/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B / 92.9MB            

data/train/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 94.9MB            

data/train/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

...

data/test/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/334323 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10471 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11013 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 'after_lines', 'is_negative', 'pr_title', 'pr_number', 'repo_name', 'repo_stars', 'repo_language', 'reviewer_username', 'author_username'],
        num_rows: 334323
    })
    validation: Dataset({
        features: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 'after_lines', 'is_negative', 'pr_title', 'pr_number', 'repo_name', 'repo_stars', 'repo_language', 'reviewer_username', 'author_username'],
        num_rows: 10471
    })
    test: Dataset({
        features: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 

In [8]:
import pandas as pd

TARGET_SIZE = 35000

df = raw['train'].to_pandas()

LANG_COL = 'language'
# Use the dataset's real label instead of inferring from blank text
NEG_COL = 'is_negative'

print("Language distribution (top 15):")
print(df[LANG_COL].value_counts().head(15))
print()
print("Negative (silent) vs. positive (commented) ratio:")
print(df[NEG_COL].value_counts(normalize=True))

Language distribution (top 15):
language
Python        82288
TypeScript    44309
Go            40091
Rust          38415
C++           33218
JavaScript    22347
C#            12523
C/C++         10741
Java           9579
C              6851
Kotlin         6150
Swift          5116
Vue            3423
PHP            3420
Shell          2864
Name: count, dtype: int64

Negative (silent) vs. positive (commented) ratio:
is_negative
False    0.782064
True     0.217936
Name: proportion, dtype: float64


In [10]:
frac = TARGET_SIZE / len(df)

sampled = (
    df.groupby([LANG_COL, NEG_COL], group_keys=False)
      .sample(frac=frac, random_state=42)
)

print(f"Sampled {len(sampled)} rows out of {len(df)} (target was {TARGET_SIZE})")
print()
print("Sampled language distribution (top 15):")
print(sampled[LANG_COL].value_counts().head(15))
print()
print("Sampled negative/positive ratio (should match the full-dataset ratio above):")
print(sampled[NEG_COL].value_counts(normalize=True))

Sampled 35002 rows out of 334323 (target was 35000)

Sampled language distribution (top 15):
language
Python        8615
TypeScript    4638
Go            4198
Rust          4022
C++           3478
JavaScript    2339
C#            1311
C/C++         1124
Java          1003
C              718
Kotlin         644
Swift          535
Vue            358
PHP            358
Shell          300
Name: count, dtype: int64

Sampled negative/positive ratio (should match the full-dataset ratio above):
is_negative
False    0.782012
True     0.217988
Name: proportion, dtype: float64


In [11]:
sampled.to_parquet(f'{PROJECT_DIR}/data/train_sample_35k.parquet', index=False)
print("Saved to", f'{PROJECT_DIR}/data/train_sample_35k.parquet')

Saved to /content/drive/MyDrive/code-reviewer-llm/data/train_sample_35k.parquet
